# Full available-data classification baseline

This notebook runs the same preparation and modeling scripts as the command line. It uses the full catalog as weak supervision, with reviewed/seed-only evaluation. `is_ai_tool` is AI detection; category is separate; no risk model is trained. See `docs/baseline_classification.md` for results and limitations.


In [ ]:
from pathlib import Path
import sys, subprocess, json
ROOT = Path.cwd()
if not (ROOT / "preprocessing_scripts").exists():
    ROOT = ROOT.parent
assert (ROOT / "preprocessing_scripts/prepare_full_classification.py").exists(), "Open from the project root or notebooks folder"
def run(*args):
    subprocess.run([sys.executable, *map(str,args)], cwd=ROOT, check=True)
print(ROOT)


Install the pinned modeling dependencies in your notebook environment if needed.

In [ ]:
run("-m", "pip", "install", "-r", ROOT / "modeling/requirements.txt")


Prepare full data. This cell deliberately overwrites only the named notebook-run output directory when rerun.

In [ ]:
DATA = ROOT / "artifacts/notebook_run/data"
MODELS = ROOT / "artifacts/notebook_run/models"
run(ROOT / "preprocessing_scripts/prepare_full_classification.py", "--out", DATA, "--overwrite")


In [ ]:
run(ROOT / "modeling/train_baseline.py", "--data", DATA, "--out", MODELS, "--overwrite")


In [ ]:
result = json.loads((MODELS / "metrics.json").read_text())
print("Selected on validation:", result["selected_binary_model"])
for name, report in result["binary"].items():
    print(name, "test accuracy:", report["test"]["accuracy"], "test macro F1:", report["test"]["macro_f1"])
print("Only four non-AI test examples; these are weak-label development results.")


In [ ]:
run(ROOT / "modeling/predict.py", "--models", MODELS, "--name", "Example Writing Assistant", "--description", "An AI assistant that generates marketing copy and drafts articles from prompts.")
